# BERT: Pre-training of Deep Bidirectional Transformers

## Learning Objectives

1. Understand masked language modeling (MLM) and bidirectional pre-training
2. Implement BERT tokenization and model loading from HuggingFace
3. Fine-tune BERT for text classification
4. Compare BERT vs. GPT (bidirectional vs. unidirectional)
5. Visualize attention patterns to understand what BERT learns

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, AutoModelForMaskedLM
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Level 1: BERT Tokenization and Basic Loading

BERT uses WordPiece tokenization and special tokens: [CLS], [SEP], [MASK], [PAD]

In [ ]:
# Load BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Example texts
texts = [
    "BERT is a bidirectional transformer model.",
    "I went to the bank to withdraw money.",
    "The cat sat on the mat."
]

# Tokenize
for text in texts:
    tokens = tokenizer.tokenize(text)
    print(f"Text: {text}")
    print(f"Tokens: {tokens}")
    print()

# Encode with special tokens
encoded = tokenizer(
    "BERT is great!",
    padding=True,
    truncation=True,
    return_tensors="pt"
)

print(f"Input IDs shape: {encoded['input_ids'].shape}")
print(f"Input IDs: {encoded['input_ids']}")
print(f"Attention mask: {encoded['attention_mask']}")
print(f"Token type IDs: {encoded['token_type_ids']}")

# Decode back
decoded = tokenizer.decode(encoded['input_ids'][0])
print(f"Decoded: {decoded}")

## Level 2: Pre-trained BERT Embeddings and Fine-tuning

Load pre-trained BERT and fine-tune for classification task

In [ ]:
# Load pre-trained BERT
bert_model = AutoModel.from_pretrained("bert-base-uncased")

# Get embeddings
inputs = tokenizer(
    "This movie is great!",
    padding=True,
    truncation=True,
    return_tensors="pt",
    max_length=128
)

with torch.no_grad():
    outputs = bert_model(**inputs)
    last_hidden = outputs.last_hidden_state  # (batch, seq_len, d_model)
    pooled_output = outputs.pooler_output  # (batch, d_model) - from [CLS] token

print(f"Last hidden state shape: {last_hidden.shape}")
print(f"Pooled output shape (from [CLS]): {pooled_output.shape}")

# Fine-tuning model for sentiment classification
classification_model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2  # Binary classification
)

# Create synthetic dataset
train_texts = [
    "This movie is amazing!",
    "Terrible film, waste of time.",
    "I loved it!",
    "Awful, don't watch it.",
    "Great performance by the actors!",
    "Very disappointing and boring."
]
train_labels = [1, 0, 1, 0, 1, 0]  # 1 = positive, 0 = negative

# Tokenize all texts
encodings = tokenizer(
    train_texts,
    padding=True,
    truncation=True,
    return_tensors="pt",
    max_length=128
)

labels_tensor = torch.tensor(train_labels)
dataset = TensorDataset(
    encodings['input_ids'],
    encodings['attention_mask'],
    labels_tensor
)

dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Fine-tune
classification_model = classification_model.to(device)
optimizer = torch.optim.Adam(classification_model.parameters(), lr=2e-5)

for epoch in range(3):
    total_loss = 0
    for batch_input_ids, batch_attention_mask, batch_labels in dataloader:
        batch_input_ids = batch_input_ids.to(device)
        batch_attention_mask = batch_attention_mask.to(device)
        batch_labels = batch_labels.to(device)
        
        outputs = classification_model(
            input_ids=batch_input_ids,
            attention_mask=batch_attention_mask,
            labels=batch_labels
        )
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}: Loss = {total_loss/len(dataloader):.4f}")

# Inference
classification_model.eval()
test_text = "This is a wonderful movie!"
test_input = tokenizer(test_text, return_tensors="pt", padding=True, truncation=True, max_length=128)

with torch.no_grad():
    test_input = {k: v.to(device) for k, v in test_input.items()}
    outputs = classification_model(**test_input)
    logits = outputs.logits
    prediction = logits.argmax(dim=-1)

print(f"\nText: {test_text}")
print(f"Predicted class: {prediction.item()} (1=positive, 0=negative)")
print(f"Confidence: {logits[0].softmax(dim=-1).max().item():.3f}")

## Real-World Example 1: Masked Language Modeling with BERT

BERT is trained on MLM objective. You can use it to fill in masked words.

In [ ]:
# Load BERT for masked language modeling
mlm_model = AutoModelForMaskedLM.from_pretrained("bert-base-uncased")
mlm_model = mlm_model.to(device)
mlm_model.eval()

# Create masked text
masked_text = "I went to the [MASK] to withdraw money."
inputs = tokenizer(masked_text, return_tensors="pt", padding=True, truncation=True, max_length=128)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Get predictions
with torch.no_grad():
    outputs = mlm_model(**inputs)
    logits = outputs.logits  # (batch, seq_len, vocab_size)

# Find [MASK] token position
mask_token_index = torch.where(inputs['input_ids'] == tokenizer.mask_token_id)[1][0]
mask_logits = logits[0, mask_token_index, :]

# Top 5 predictions
top_5_indices = torch.topk(mask_logits, 5).indices

print(f"Text: {masked_text}")
print(f"\nTop 5 predictions for [MASK]:")
for i, token_id in enumerate(top_5_indices):
    word = tokenizer.decode([token_id.item()])
    score = mask_logits[token_id].item()
    print(f"  {i+1}. {word:15s} (score: {score:.3f})")

print("\n(Notice BERT predicts 'bank' - it's bidirectional and sees the context to the right!)")

## Real-World Example 2: Named Entity Recognition with BERT

Fine-tune BERT for per-token classification (NER)

In [ ]:
from transformers import AutoModelForTokenClassification

# NER model: classify each token as person (0), location (1), organization (2), or O (3)
ner_model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)
ner_model = ner_model.to(device)

# Dummy NER dataset: sentences with per-token labels
ner_texts = [
    "John works at Microsoft in Seattle.",
    "Apple is a company in California."
]
# Per-token labels (O, person, organization, location, etc.)
# Simplified: just label the main entities

# Tokenize and align labels
tokenized = tokenizer(
    ner_texts[0],
    padding=True,
    truncation=True,
    return_tensors="pt",
    max_length=128
)

# Create dummy labels (0=O, 1=Person, 2=Org, 3=Location)
# [CLS] John works at Microsoft in Seattle [SEP] [PAD] ...
dummy_labels = torch.zeros(tokenized['input_ids'].shape[0], tokenized['input_ids'].shape[1], dtype=torch.long)
# Manually label some tokens (just for demo)
dummy_labels[0, 1] = 1  # John -> person
dummy_labels[0, 4] = 2  # Microsoft -> org
dummy_labels[0, 6] = 3  # Seattle -> location

# Quick training step
ner_model.train()
tokenized = {k: v.to(device) for k, v in tokenized.items()}
dummy_labels = dummy_labels.to(device)

outputs = ner_model(
    input_ids=tokenized['input_ids'],
    attention_mask=tokenized['attention_mask'],
    labels=dummy_labels
)

print(f"NER Model Loss: {outputs.loss.item():.4f}")
print(f"NER logits shape: {outputs.logits.shape}")  # (batch, seq_len, num_labels)

## Real-World Example 3: Sentence Similarity with BERT

Use BERT embeddings to compute semantic similarity between sentences

In [ ]:
from torch.nn.functional import cosine_similarity

# Load BERT base model for embedding
bert_embedder = AutoModel.from_pretrained("bert-base-uncased")
bert_embedder = bert_embedder.to(device)
bert_embedder.eval()

# Sentence pairs
sentence_pairs = [
    ("The cat is sitting on the mat.", "A feline is resting on the rug."),  # Semantically similar
    ("The cat is sitting on the mat.", "I like pizza."),  # Not similar
    ("BERT is a transformer model.", "BERT is based on the transformer architecture."),  # Very similar
]

def get_bert_embedding(text):
    """Get [CLS] token embedding as sentence representation"""
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = bert_embedder(**inputs)
        # Use [CLS] token embedding (index 0)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
    
    return cls_embedding

# Compute similarities
print("Sentence Similarity using BERT:")
print()

for sent1, sent2 in sentence_pairs:
    emb1 = get_bert_embedding(sent1)
    emb2 = get_bert_embedding(sent2)
    
    similarity = cosine_similarity(emb1, emb2).item()
    
    print(f"Sent1: {sent1}")
    print(f"Sent2: {sent2}")
    print(f"Similarity: {similarity:.4f}")
    print()

## Key Takeaways

**BERT Mechanism:**
- Bidirectional: sees context left AND right
- Pre-training: masked language modeling (MLM) + next sentence prediction (NSP)
- [CLS] token: aggregates sequence information for classification
- Fine-tuning: add task-specific head, train for 3-5 epochs

**Key Tasks:**
- Classification: use [CLS] token output + linear layer
- NER/sequence labeling: use per-token outputs
- Semantic similarity: use [CLS] embeddings
- MLM: predict masked tokens (BERT's pre-training task)

**When to use BERT:**
- Understanding tasks (classification, NER, QA)
- Limited labeled data (few-shot after fine-tuning)
- Production NLP systems (stable, pre-trained)

**Related papers:**
- [Attention Is All You Need](./01-attention-is-all-you-need.md) - Transformer foundation
- [GPT-3](./03-gpt3.md) - Autoregressive alternative
- [LoRA](./05-lora.md) - Efficient fine-tuning